In [2]:
import pandas as pd
import duckdb
import os
import numpy as np


conn = duckdb.connect('../database/olist.db')

# JOINS

##### Write a query joining olist_orders and olist_customers to return all orders alongside the customer's state. Then explain — what happens to orders where the customer_id exists in orders but not in customers? Which JOIN type protects you from silent data loss here?

In [12]:
query = """ 

    select
        o.order_id, o.customer_id, o.order_status, o.order_purchase_timestamp,
        c.customer_state, c.customer_city,
        CASE 
            WHEN c.customer_id IS NULL then 'orphaned'
            else 'matched' 
        END as record_quality_flag
    from orders o
    LEFT JOIN customers c
    on o.customer_id = c.customer_id


"""

conn.execute(query).df()

,order_id,customer_id,order_status,order_purchase_timestamp,customer_state,customer_city,record_quality_flag
0,e481f51cbdc54678b7cc49136f2d6af7,9ef432eb6251297304e76186b10a928d,delivered,2017-10-02 10:56:33,SP,sao paulo,matched
1,53cdb2fc8bc7dce0b6741e2150273451,b0830fb4747a6c6d20dea0b8c802d7ef,delivered,2018-07-24 20:41:37,BA,barreiras,matched
2,47770eb9100c2d0c44946d9cf07ec65d,41ce2a54c0b03bf3443c3d931a367089,delivered,2018-08-08 08:38:49,GO,vianopolis,matched
3,ad21c59c0840e6cb83a9ceb5573f8159,8ab97904e6daea8866dbdbc4fb7aad2c,delivered,2018-02-13 21:18:39,SP,santo andre,matched
4,a4591c265e18cb1dcee52889e2d8acc3,503740e9ca751ccdda7ba28e9ab8f608,delivered,2017-07-09 21:57:05,PR,congonhinhas,matched
...,...,...,...,...,...,...,...
99436,9c5dedf39a927c1b2549525ed64a053c,39bd1228ee8140590ac3aca26f2dfe00,delivered,2017-03-09 09:54:05,SP,sao jose dos campos,matched
99437,63943bddc261676b46f01ca7ac2f7bd8,1fca14ff2861355f6e5f14306ff977a7,delivered,2018-02-06 12:58:58,SP,praia grande,matched
99438,83c1379a015df1e13d02aae0204711ab,1aa71eb042121263aafbe80c1b562c9c,delivered,2017-08-27 14:46:43,BA,nova vicosa,matched
99439,11c177c8e97725db2631073c19f07b62,b331b74b18dc79bcdf6532d51e1637c1,delivered,2018-01-08 21:28:27,RJ,japuiba,matched


In [15]:
query = """ 

    -- How bad is the data quality problem?
    SELECT
        COUNT(*)                                        AS total_orders,
        COUNT(c.customer_id)                            AS matched_orders,
        COUNT(*) - COUNT(c.customer_id)                 AS orphaned_orders,
        ROUND(
            (COUNT(*) - COUNT(c.customer_id)) * 100.0 
            / COUNT(*), 2)                              AS orphaned_pct

    FROM orders o
    LEFT JOIN customers c
        ON o.customer_id = c.customer_id


"""

conn.execute(query).df()

,total_orders,matched_orders,orphaned_orders,orphaned_pct
0,99441,99441,0,0.0


##### Join olist_orders, olist_order_items and olist_products to return order_id, product_category_name and total price per order. A junior analyst used INNER JOIN throughout — what records are they silently losing and why does it matter?

In [20]:
query = """ 

    select
        o.order_id, count(p.product_category_name) as total_categories, round(sum(oi.price),2) as total_price
    from orders o
    LEFT JOIN order_items oi
    on o.order_id = oi.order_id
    LEFT JOIN products p
    on oi.product_id = p.product_id
    group by o.order_id

"""

conn.execute(query).df()

,order_id,total_categories,total_price
0,0028de0ca693a1bb26448916a81105cc,1,29.99
1,00611822267e76e0055c25c18506f06e,1,159.90
2,0068e836900867da359bd81db9227a33,1,96.99
3,006dd93155bc2abd844cc5eed3a0fe7f,1,49.90
4,00ab9fa91aafa3d881164b0da48999aa,1,579.00
...,...,...,...
99436,6d9bd9b4caa7d961a6d31427ad0e557e,0,NaN
99437,801bda1bc87e8484430a1ad75f51f128,0,NaN
99438,261e56a43de07e1b1bc9e8615c8e8aa9,0,NaN
99439,7018a2cb4802de2f9ed4b09b6a98e6fd,0,NaN


##### Return all sellers from olist_sellers who have never received any order in olist_order_items. Write this using three different approaches — LEFT JOIN with NULL check, NOT EXISTS and NOT IN. Explain which is safest and why.

In [ ]:
# Left join NULL CHECKS
query = """ 

    select
       s.seller_id, s.seller_state, s.seller_city, oi.order_id
    from sellers s
    LEFT JOIN order_items oi
    on s.seller_id = oi.seller_id
    WHERE oi.seller_id IS NULL
     

"""

conn.execute(query).df()
# NOT EXSITS
query = """ 

    select
       s.seller_id, s.seller_state, s.seller_city
    from sellers s
    where NOT EXISTS (
        SELECT 1
        FROM order_items oi
        where oi.seller_id = s.seller_id
    )
     

"""

conn.execute(query).df()
# NOT IN
query = """ 

    select
       s.seller_id, s.seller_state, s.seller_city
    from sellers s
    LEFT JOIN order_items oi
    on s.seller_id = oi.seller_id
    WHERE s.seller_id NOT IN (
        select seller_id
        from order_items
        where seller_id is not null
    )
     

"""

conn.execute(query).df()

,seller_id,seller_state,seller_city


##### Perform a FULL OUTER JOIN between olist_customers and olist_orders. Identify — customers with no orders, orders with no matching customer and explain under what real-world data scenarios each of these anomalies occurs.

In [42]:
query = """ 

    SELECT
        -- Coalesce to handle NULLs from either side
        COALESCE(c.customer_id, 'No Customer Record')   AS customer_id,
        COALESCE(o.order_id,    'No Order Record')       AS order_id,
        c.customer_city,
        c.customer_state,
        o.order_status,
        o.order_purchase_timestamp,

        -- Classify every row into its bucket
        CASE
            WHEN c.customer_id IS NULL THEN 'Orphaned Order'
            WHEN o.customer_id IS NULL THEN 'Customer Without Order'
            ELSE 'Matched Record'
        END                                             AS record_type

    FROM customers c
    FULL OUTER JOIN orders o
        ON c.customer_id = o.customer_id

    -- Optional: filter to only anomalies
    WHERE c.customer_id IS NULL
    OR o.customer_id IS NULL

"""

conn.execute(query).df()

,customer_id,order_id,customer_city,customer_state,order_status,order_purchase_timestamp,record_type


##### A colleague joins olist_orders to olist_order_payments using a LEFT JOIN and then filters on payment_type = 'credit_card' in the WHERE clause. The result silently converts the LEFT JOIN into an INNER JOIN. Explain exactly why this happens and rewrite it correctly.

In [56]:
query = """ 

    SELECT
       o.order_id, p.payment_type,
       CASE 
        when p.payment_type IS NULL then 'No Payment'
        when p.payment_type = 'credit_card' then 'Credit card order' 
            else 'Other Payment'
        END as record_type
    FROM orders o
    LEFT JOIN order_payments p
    on o.order_id = p.order_id
        and p.payment_type = 'credit_card'
"""

conn.execute(query).df()

,order_id,payment_type,record_type
0,e481f51cbdc54678b7cc49136f2d6af7,credit_card,Credit card order
1,47770eb9100c2d0c44946d9cf07ec65d,credit_card,Credit card order
2,949d5b44dbf5de918fe9c16f97b45f8a,credit_card,Credit card order
3,ad21c59c0840e6cb83a9ceb5573f8159,credit_card,Credit card order
4,a4591c265e18cb1dcee52889e2d8acc3,credit_card,Credit card order
...,...,...,...
99726,5fabc81b6322c8443648e1b21a6fef21,None,No Payment
99727,788541a19c0791de0504c5a9cb7e7bd5,None,No Payment
99728,f9e3402be5a5ea63344347582ca9f45f,None,No Payment
99729,38e9133ce29f6bbe35aed9c3863dce01,None,No Payment


##### You are joining olist_orders (5M rows) to olist_order_items (10M rows) to olist_order_payments (6M rows). After the join your result set has significantly more rows than olist_orders. Diagnose exactly what is happening, why it happens and write the corrected query. This is the fan trap — explain it clearly as you would to a junior analyst.

# SELF JOINs & Hierarchical Problems

##### Using a SELF JOIN on olist_customers, find all pairs of customers who live in the same city and state. Return customer_id_1, customer_id_2, city and state. Make sure you don't return duplicate pairs i.e. (A,B) and (B,A).

##### Using a SELF JOIN on olist_order_reviews, find all cases where the same customer left reviews with a score difference of 3 or more points across two different orders. This could signal highly inconsistent customer experience.

##### Using a SELF JOIN on olist_order_items, identify products that were ordered together in the same order — i.e. appear in the same order_id. Return product_id_1, product_id_2 and co-occurrence count. Filter for pairs that appeared together more than 10 times.

##### Simulate a seller peer comparison using a SELF JOIN. For each seller, find all other sellers in the same state and compare their total revenue, average review score and on-time delivery rate. Return each seller alongside their state average for each metric and flag sellers who are below state average on all three as 'Underperformer'.